In [2]:
# reading the file
with open('hostel_bois.txt', 'r', encoding='utf-8') as f:
    content = f.read()

In [24]:
messages= []
system_messages = 0
media_omitted = 0
deleted_messages = 0

# for keeping track of the dates and names
senders = set()
dates = set()

with open('hostel_bois.txt', 'r', encoding='utf-8') as file:
    lines = file.readlines()
for line in lines:
    cleaned_line = line.strip()

    # skipping lines that are empty
    if cleaned_line=="":
        continue
    # so this(next part) is ai assisted im sorry im dumb but im trying
    # i still dont understand why ai did that 

    if len(cleaned_line) >= 16 and cleaned_line[2] == '/' and cleaned_line[5] == '/' and cleaned_line[12] == ':':
        if ' - ' in cleaned_line and ':' not in cleaned_line.split(' - ', 1)[1]:
            system_messages += 1
            continue
        # so here i will seperate timestamp from message that are in that txt file
        time_part, details_part = cleaned_line.split(' - ', 1)
       
        # and now i will pick dates from timestamp just the starting 8 characters i believe

        date_part = time_part[:8]
        dates.add(date_part)

        # now i ll seperate the sender's name
        sender_name, message_text = details_part.split(':', 1)
        # cleaning whitespaces
        sender_name = sender_name.strip()
        message_text = message_text.strip()
        senders.add(sender_name)

        if message_text=="<Media omitted>":
            media_omitted+=1
        if message_text=="This message was deleted":
            deleted_messages+=1
        # creating a dictionary now 

        message_dictionary={
            "timestamp": time_part,
            "sender":sender_name,
            "text":message_text
        }
        messages.append(message_dictionary)
        # adding the dictionary to list because i cant create list for timestamp because  i have to take keys and thier value
    else:
        if len(messages)>0:
            messages[-1]["text"]=messages[-1]["text"] + " " + cleaned_line
            
print(f"successfully parsed {len(messages)} message from {len(senders)} participants over {len(dates)} days ,skipped {system_messages} system message, {media_omitted} media-omitted, {deleted_messages} deleted messages. ")
  #(AI ASSISTED)i have to check the print line from ai  .
        # after trying 23 times it happened 
# feature 1 completed 



successfully parsed 3174 message from 6 participants over 60 days ,skipped 4 system message, 32 media-omitted, 15 deleted messages. 


In [30]:
# now for feature 2 im creating a empty dictionary name as count

message_count={}
for msg in messages:
    name=msg["sender"]
    if name in message_count:         # so what im doing here is simply getting loop for messages to extract out
                              # the senders names from parsed messages that ive done 
        message_count[name]+=1
    else:
        message_count[name]=1

total_messages = len(messages)

sorted_members = sorted(message_count.items(), key=lambda x: x[1], reverse=True)


print("========================================================")
print("GROUP OVERVIEW")
print("========================================================")
print("group          : Hostel Bois")
print("period         : 01 April 2024 to 30 May 2024 (60 days)")
print("total messages : " + str(total_messages))
print("participants   : " + str(len(senders)))
print("\nmessage per person")
for name, count in sorted_members: 
    percentage = (count / total_messages) * 100
    print(f"{name:<10} : {count:<5} ({percentage:.1f}%)")

# so this upper line will sort(by countof message ) the biggest number like reversing default so that who spams the most message comes first 


GROUP OVERVIEW
group          : Hostel Bois
period         : 01 April 2024 to 30 May 2024 (60 days)
total messages : 3174
participants   : 6

message per person
Rahul      : 953   (30.0%)
Priya      : 718   (22.6%)
Neha       : 635   (20.0%)
Aman       : 490   (15.4%)
Karan      : 354   (11.2%)
Vikas      : 24    (0.8%)


In [36]:
# feature 3 ,able to tell the most active hours and days

date_count={}
hour_count={}

for msg in messages:
    timestamp=msg["timestamp"]
    msg_date=timestamp[:8]#extracting the date of messages
    msg_hour=timestamp[10:14]

    if msg_date in date_count:
        date_count[msg_date]+=1
    else:
        date_count[msg_date]=1

    if msg_hour in hour_count:
        hour_count[msg_hour]+=1
    else:
        hour_count[msg_hour]=1
        
bussiest_date=""  #finding the date with most counts so to determine the bussiest day for feature 3
max_date_count=0

for msg_date in date_count:
    if date_count[msg_date]>max_date_count: 
        max_date_count=date_count[msg_date]
        bussiest_date=msg_date
        
bussiest_hour=""  #finding the hour with most counts so to determine the bussiest hour for feature 3
max_hour_count=0

for msg_hour in hour_count:
    if hour_count[msg_hour]>max_hour_count:
        max_hour_count=hour_count[msg_hour]
        bussiest_hour=msg_hour
print("MOST ACTIVE TIME PATTERNS")
print("busiest day  : "+bussiest_date+" (" + str(max_date_count) + " messages)")
print("busiest hour : "+ bussiest_hour+":00 (" + str(max_hour_count) + " total messages)")

    

MOST ACTIVE TIME PATTERNS
busiest day  : 04/05/24 (76 messages)
busiest hour : 18:5:00 (63 total messages)


In [44]:
# feature 4 
# so i donot know how to create heatmapes from numpy sorry !,im using help from ai 
# AI assisted!!
import numpy as np
# turn unique senders set into a sorted list so the order never changes
unique_senders_list = sorted(list(senders))
#create a 6 by 24 matrix of zeros using numpy
heatmap_matrix = np.zeros((6, 24))
# fill the matrix by looping through parsed messages
for msg in messages:
    sender = msg["sender"]
    timestamp = msg["timestamp"] 
    # extract the hour string and turn it into an integer index (0 to 23)
    hour = int(timestamp[10:12])
    # find the row index of this sender in  list (0 to 5)
    row_index = unique_senders_list.index(sender)
    
# increment that specific cell in the numpy matrix
    heatmap_matrix[row_index, hour]+=1
#printing heatmap report header
print("ACTIVITY HEATMAP( by hour)")

# Print the hour column headers
print("           00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23")

# loop through each person to render their shading row manually
for i in range(6):
    name = unique_senders_list[i]
    row_string = ""  
 # find the highest number of messages this specific person sent in any hour
    person_max = np.max(heatmap_matrix[i])
# avoid dividing by zero if someone has 0 messages
    if person_max==0:
        person_max=1
    for hour in range(24):
        value=heatmap_matrix[i, hour]
        # calculate the percentage relative to their personal maximum hour
        percentage = (value / person_max) * 100
     # Assign shading symbols based on the four required rubric thresholds
        if percentage ==0:
            symbol = " . "
        elif percentage <= 25:
            symbol = " ░ "
        elif percentage <= 50:
            symbol = " ▒ "
        elif percentage <= 75:
            symbol = " ▓ "
        else:
            symbol = " █ "
            
        row_string += symbol# print their name left-aligned next to their symbol stream
    print(f"{name:<10} {row_string}")


ACTIVITY HEATMAP(messages by hour)
           00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23
Aman        ▓  █  ▓  ▓  █  .  .  .  .  .  .  .  .  .  ░  ░  ░  ░  ░  ░  ░  ░  .  ▓ 
Karan       .  .  .  .  .  .  .  ░  ▒  ▒  ▓  ▒  █  ▓  █  ▓  ▓  ▓  ▓  █  ▓  ▒  ░  ░ 
Neha        .  .  .  .  .  ▒  ░  ░  ▓  █  █  ▒  ▓  ▓  ▒  ░  ▓  █  █  █  ▓  ▒  ▒  ▒ 
Priya       .  .  .  .  .  .  ░  ▒  ▓  █  █  █  █  ▓  ▓  ▒  ▒  ▓  ▓  █  ▓  ▒  ▒  ░ 
Rahul       ░  ░  ░  ░  ░  ░  ░  ░  ░  ░  ░  ░  ▓  ▒  ▒  ▓  ▓  ▒  █  ▓  ▒  █  ▓  ▓ 
Vikas       .  .  .  .  .  .  .  ▒  █  ▒  ▒  .  ▓  ▓  .  ▒  ▒  █  ▓  ▓  ▒  ▒  ▒  ▓ 


In [54]:
# feature 5
#sorting the words by thier count

def get_word_count(item):
    return item[1]
stop_words=['this','i', 'is', 'the', 'a', 'and', 'or', 'to', 'of', 'in', 'on', 'for','there','that','was','how','that','so'] #took the words from pdf of minor project
word_counts={} #creating a dictionary
# skipping media omiited and deleted messages
for msg in messages:
    if msg["text"]=="<Media ommited>" or msg["text"]=="This message was deleted": 
        continue
    words=msg["text"].lower().split() #eliminating the cases like caps letters
    for word in words:
        word = word.strip(",.;!:?/")
        if word == "" or word in stop_words:
            continue #removing the unnecesary symbols
        if word in word_counts:
            word_counts[word] += 1
        else:
            word_counts[word] = 1
sorted_words= sorted(word_counts.items(),key=get_word_count,reverse=True)
top_10_words= sorted_words[:10]
print("group's favourite words")

# so this loop is for creating the bar graph effect by taking shading from pdf itself

for word, count in top_10_words:
    bar_length = count // 25
    bar_visual = "█" * bar_length
    print(f"{word:<12} : {count:<5} {bar_visual}")


group's favourite words
guys         : 318   ████████████
you          : 312   ████████████
about        : 274   ██████████
hai          : 268   ██████████
am           : 260   ██████████
today        : 257   ██████████
at           : 257   ██████████
my           : 223   ████████
he           : 220   ████████
his          : 217   ████████


In [55]:
# alright i have to combine whole code for the preview upto feature 5 because i have been able to do until feature 5



In [56]:
messages= []
system_messages = 0
media_omitted = 0
deleted_messages = 0

# for keeping track of the dates and names
senders = set()
dates = set()

with open('hostel_bois.txt', 'r', encoding='utf-8') as file:
    lines = file.readlines()
for line in lines:
    cleaned_line = line.strip()

    # skipping lines that are empty
    if cleaned_line=="":
        continue
    # so this(next part) is ai assisted im sorry im dumb but im trying
    # i still dont understand why ai did that 

    if len(cleaned_line) >= 16 and cleaned_line[2] == '/' and cleaned_line[5] == '/' and cleaned_line[12] == ':':
        if ' - ' in cleaned_line and ':' not in cleaned_line.split(' - ', 1)[1]:
            system_messages += 1
            continue
        # so here i will seperate timestamp from message that are in that txt file
        time_part, details_part = cleaned_line.split(' - ', 1)
       
        # and now i will pick dates from timestamp just the starting 8 characters i believe

        date_part = time_part[:8]
        dates.add(date_part)

        # now i ll seperate the sender's name
        sender_name, message_text = details_part.split(':', 1)
        # cleaning whitespaces
        sender_name = sender_name.strip()
        message_text = message_text.strip()
        senders.add(sender_name)

        if message_text=="<Media omitted>":
            media_omitted+=1
        if message_text=="This message was deleted":
            deleted_messages+=1
        # creating a dictionary now 

        message_dictionary={
            "timestamp": time_part,
            "sender":sender_name,
            "text":message_text
        }
        messages.append(message_dictionary)
        # adding the dictionary to list because i cant create list for timestamp because  i have to take keys and thier value
    else:
        if len(messages)>0:
            messages[-1]["text"]=messages[-1]["text"] + " " + cleaned_line
            
print(f"successfully parsed {len(messages)} message from {len(senders)} participants over {len(dates)} days ,skipped {system_messages} system message, {media_omitted} media-omitted, {deleted_messages} deleted messages. ")
  #(AI ASSISTED)i have to check the print line from ai  .
        # after trying 23 times it happened 
# feature 1 completed 

# now for feature 2 im creating a empty dictionary name as count

message_count={}
for msg in messages:
    name=msg["sender"]
    if name in message_count:         # so what im doing here is simply getting loop for messages to extract out
                              # the senders names from parsed messages that ive done 
        message_count[name]+=1
    else:
        message_count[name]=1

total_messages = len(messages)

sorted_members = sorted(message_count.items(), key=lambda x: x[1], reverse=True)


print("========================================================")
print("GROUP OVERVIEW")
print("========================================================")
print("group          : Hostel Bois")
print("period         : 01 April 2024 to 30 May 2024 (60 days)")
print("total messages : " + str(total_messages))
print("participants   : " + str(len(senders)))
print("\nmessage per person")
for name, count in sorted_members: 
    percentage = (count / total_messages) * 100
    print(f"{name:<10} : {count:<5} ({percentage:.1f}%)")

# so this upper line will sort(by countof message ) the biggest number like reversing default so that who spams the most message comes first 



# feature 3 ,able to tell the most active hours and days

date_count={}
hour_count={}

for msg in messages:
    timestamp=msg["timestamp"]
    msg_date=timestamp[:8]#extracting the date of messages
    msg_hour=timestamp[10:14]

    if msg_date in date_count:
        date_count[msg_date]+=1
    else:
        date_count[msg_date]=1

    if msg_hour in hour_count:
        hour_count[msg_hour]+=1
    else:
        hour_count[msg_hour]=1
        
bussiest_date=""  #finding the date with most counts so to determine the bussiest day for feature 3
max_date_count=0

for msg_date in date_count:
    if date_count[msg_date]>max_date_count: 
        max_date_count=date_count[msg_date]
        bussiest_date=msg_date
        
bussiest_hour=""  #finding the hour with most counts so to determine the bussiest hour for feature 3
max_hour_count=0

for msg_hour in hour_count:
    if hour_count[msg_hour]>max_hour_count:
        max_hour_count=hour_count[msg_hour]
        bussiest_hour=msg_hour
print("MOST ACTIVE TIME PATTERNS")
print("busiest day  : "+bussiest_date+" (" + str(max_date_count) + " messages)")
print("busiest hour : "+ bussiest_hour+":00 (" + str(max_hour_count) + " total messages)")

# feature 4 
# so i donot know how to create heatmapes from numpy sorry !,im using help from ai 
# AI assisted!!
import numpy as np
# turn unique senders set into a sorted list so the order never changes
unique_senders_list = sorted(list(senders))
#create a 6 by 24 matrix of zeros using numpy
heatmap_matrix = np.zeros((6, 24))
# fill the matrix by looping through parsed messages
for msg in messages:
    sender = msg["sender"]
    timestamp = msg["timestamp"] 
    # extract the hour string and turn it into an integer index (0 to 23)
    hour = int(timestamp[10:12])
    # find the row index of this sender in  list (0 to 5)
    row_index = unique_senders_list.index(sender)
    
# increment that specific cell in the numpy matrix
    heatmap_matrix[row_index, hour]+=1
#printing heatmap report header
print("ACTIVITY HEATMAP( by hour)")

# Print the hour column headers
print("           00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23")

# loop through each person to render their shading row manually
for i in range(6):
    name = unique_senders_list[i]
    row_string = ""  
 # find the highest number of messages this specific person sent in any hour
    person_max = np.max(heatmap_matrix[i])
# avoid dividing by zero if someone has 0 messages
    if person_max==0:
        person_max=1
    for hour in range(24):
        value=heatmap_matrix[i, hour]
        # calculate the percentage relative to their personal maximum hour
        percentage = (value / person_max) * 100
     # Assign shading symbols based on the four required rubric thresholds
        if percentage ==0:
            symbol = " . "
        elif percentage <= 25:
            symbol = " ░ "
        elif percentage <= 50:
            symbol = " ▒ "
        elif percentage <= 75:
            symbol = " ▓ "
        else:
            symbol = " █ "
            
        row_string += symbol# print their name left-aligned next to their symbol stream
    print(f"{name:<10} {row_string}")
# feature 5
#sorting the words by thier count

def get_word_count(item):
    return item[1]
stop_words=['this','i', 'is', 'the', 'a', 'and', 'or', 'to', 'of', 'in', 'on', 'for','there','that','was','how','that','so'] #took the words from pdf of minor project
word_counts={} #creating a dictionary
# skipping media omiited and deleted messages
for msg in messages:
    if msg["text"]=="<Media ommited>" or msg["text"]=="This message was deleted": 
        continue
    words=msg["text"].lower().split() #eliminating the cases like caps letters
    for word in words:
        word = word.strip(",.;!:?/")
        if word == "" or word in stop_words:
            continue #removing the unnecesary symbols
        if word in word_counts:
            word_counts[word] += 1
        else:
            word_counts[word] = 1
sorted_words= sorted(word_counts.items(),key=get_word_count,reverse=True)
top_10_words= sorted_words[:10]
print("group's favourite words")

# so this loop is for creating the bar graph effect by taking shading from pdf itself

for word, count in top_10_words:
    bar_length = count // 25
    bar_visual = "█" * bar_length
    print(f"{word:<12} : {count:<5} {bar_visual}")



successfully parsed 3174 message from 6 participants over 60 days ,skipped 4 system message, 32 media-omitted, 15 deleted messages. 
GROUP OVERVIEW
group          : Hostel Bois
period         : 01 April 2024 to 30 May 2024 (60 days)
total messages : 3174
participants   : 6

message per person
Rahul      : 953   (30.0%)
Priya      : 718   (22.6%)
Neha       : 635   (20.0%)
Aman       : 490   (15.4%)
Karan      : 354   (11.2%)
Vikas      : 24    (0.8%)
MOST ACTIVE TIME PATTERNS
busiest day  : 04/05/24 (76 messages)
busiest hour : 18:5:00 (63 total messages)
ACTIVITY HEATMAP( by hour)
           00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23
Aman        ▓  █  ▓  ▓  █  .  .  .  .  .  .  .  .  .  ░  ░  ░  ░  ░  ░  ░  ░  .  ▓ 
Karan       .  .  .  .  .  .  .  ░  ▒  ▒  ▓  ▒  █  ▓  █  ▓  ▓  ▓  ▓  █  ▓  ▒  ░  ░ 
Neha        .  .  .  .  .  ▒  ░  ░  ▓  █  █  ▒  ▓  ▓  ▒  ░  ▓  █  █  █  ▓  ▒  ▒  ▒ 
Priya       .  .  .  .  .  .  ░  ▒  ▓  █  █  █  █  ▓  ▓  ▒  ▒  ▓  ▓  █  ▓  ▒